In [0]:
import requests
from pyspark.sql.types import *
from datetime import date
from delta.tables import DeltaTable

In [0]:
CATALOG = "file_upload"
SCHEMA  = "silver"
TABLE   = "ref_exchange_rates"

currencies = ["EUR", "HUF", "USD"]
rates = []

for currency in currencies:
    url  = f"https://api.nbp.pl/api/exchangerates/rates/a/{currency}/?format=json"
    resp = requests.get(url)
    resp.raise_for_status()
    
    data     = resp.json()
    rate     = data["rates"][0]["mid"]
    eff_date = data["rates"][0]["effectiveDate"]
    
    rates.append({
        "FROM_CUR":       currency,
        "TO_CUR":         "PLN",
        "EXCHG_RATE":     float(rate),
        "EFFECTIVE_DATE": eff_date,
        "FISCVARNT":      "R2",
        "FISCYEAR":       str(date.today().year),
        "loaded_at":      str(date.today())
    })
    print(f"{currency}/PLN: {rate} (data: {eff_date})")

    schema = StructType([
    StructField("FROM_CUR",       StringType()),
    StructField("TO_CUR",         StringType()),
    StructField("EXCHG_RATE",     DoubleType()),
    StructField("EFFECTIVE_DATE", StringType()),
    StructField("FISCVARNT",      StringType()),
    StructField("FISCYEAR",       StringType()),
    StructField("loaded_at",      StringType()),
])

df = spark.createDataFrame(rates, schema)

if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{TABLE}"):
    delta_table = DeltaTable.forName(spark, f"{CATALOG}.{SCHEMA}.{TABLE}")
    
    delta_table.alias("existing") \
        .merge(
            df.alias("new"),
            "existing.FROM_CUR = new.FROM_CUR AND existing.EFFECTIVE_DATE = new.EFFECTIVE_DATE"
        ) \
        .whenNotMatchedInsertAll() \
        .execute()
    
    print("Kursy zaktualizowane przez merge")
else:
    df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")
    
    print("Tabela utworzona, kursy załadowane")

print(f"Przetworzono {len(rates)} waluty")

Generowanie tabeli z kalendarzem dni roboczych, biblioteka holidays

In [0]:
%pip install holidays
import holidays
import calendar
from datetime import date
from pyspark.sql.types import *

In [0]:
# Generuj dni robocze per miesiąc
current_year = date.today().year
pl_holidays = holidays.Poland(years=range(current_year, current_year+2))
rows = []

for year in range(current_year, current_year+2):
    for month in range(1, 13):
        _, last_day = calendar.monthrange(year, month)
        working_days = 0
        for day in range(1, last_day + 1):
            d = date(year, month, day)
            if d.weekday() < 5 and d not in pl_holidays:
                working_days += 1
        rows.append({
            "CALYEAR":       str(year),
            "CALMONTH":      f"{year}-{month:02d}",
            "WORKING_DAYS":  working_days
        })

schema = StructType([
    StructField("CALYEAR",      StringType()),
    StructField("CALMONTH",     StringType()),
    StructField("WORKING_DAYS", IntegerType()),
])

df_cal = spark.createDataFrame(rows, schema)

df_cal.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("file_upload.silver.ref_calendar")

print(f"Kalendarz załadowany: {df_cal.count()} miesięcy")